# 考勤系统完整工作流介绍

本notebook详细介绍基于人脸识别的班级考勤系统的完整工作流。系统采用Flask后端、浏览器前端的BS架构，集成了人脸识别、活体检测、情绪分析等功能。

## 系统整体概述

### 业务场景
班级考勤系统主要应用于学校或培训机构，实现自动化考勤管理。通过摄像头实时采集学生人脸图像，系统自动识别身份、验证活体、分析情绪，并记录考勤信息。

### 核心工作流程
1. **用户登录**：教师/学生通过账号密码登录系统
2. **学生注册**：教师上传学生照片，建立人脸特征库
3. **实时考勤**：
   - 前端采集视频流
   - 后端进行活体检测、人脸识别、情绪分析
   - 写入考勤记录
4. **合照考勤**：上传活动合照，批量识别参与学生
5. **数据统计**：查看考勤记录、情绪分布、活动参与情况

### 主要功能特性
- **基础考勤**：实时摄像头采集，人脸1:N匹配，写入考勤记录，支持按日期/学号筛选 + Excel导出
- **合照识别**：上传大合照（10–50人），后端批量人脸检测 → 匹配学生 → 自动生成参与名单与活动频次统计（柱状图/表格）
- **情绪分析**：考勤、合照流程同步调用DeepFace做情绪识别（7类），后端落库，前端饼图 + 时间折线
- **安全防护**：随机一次性活体挑战（1~2个动作），抗静态照片/重放视频；教师/学生角色权限隔离；上传大小限制32MB；模型懒加载

### 技术栈
- **后端**：Python Flask框架
- **前端**：HTML/CSS/JavaScript + Chart.js
- **数据库**：SQLite
- **AI模型**：dlib (人脸检测/识别), DeepFace (情绪分析), PyTorch (活体检测), GFPGAN (超分)

## 整体架构

### 前后端架构
- **前端 (Browser)**：负责用户界面、摄像头控制、实时视频采集
  - 技术栈：HTML5 Canvas, JavaScript, Chart.js
  - 功能：登录界面、考勤页面、学生管理、统计图表
- **后端 (Flask)**：提供REST API接口，处理业务逻辑
  - 技术栈：Python Flask, Werkzeug
  - 功能：用户认证、数据处理、模型推理
- **数据库 (SQLite)**：轻量级关系型数据库
  - 表结构：users(用户), students(学生), attendance_records(考勤), activities(活动)
- **模型层**：AI算法实现
  - dlib：人脸检测、关键点、特征提取
  - DeepFace：情绪识别
  - PyTorch：活体检测模型
  - GFPGAN：人脸超分

### 系统组件图

```
┌─────────────────┐    HTTP/HTTPS    ┌─────────────────┐
│   Browser       │◄────────────────►│   Flask Server  │
│   Frontend      │                  │   Backend       │
│                 │                  │                 │
│ • HTML/CSS/JS   │                  │ • REST API      │
│ • Chart.js      │                  │ • Business Logic│
│ • Camera API    │                  │ • Model Serving │
└─────────────────┘                  └─────────────────┘
         │                                   │
         │                                   │
         ▼                                   ▼
┌─────────────────┐                  ┌─────────────────┐
│   SQLite DB     │◄────────────────►│   AI Models     │
│                 │                  │                 │
│ • users         │                  │ • dlib          │
│ • students      │                  │ • DeepFace      │
│ • attendance    │                  │ • Clarityeye    │
│ • activities    │                  │ • GFPGAN        │
└─────────────────┘                  └─────────────────┘
```

### 数据流图

```
摄像头帧 → 前端采集 → HTTP POST → 后端接收 → 图像解码 → 人脸检测 → 特征提取 → 数据库匹配 → 活体验证 → 情绪分析 → 结果返回 → 前端显示
```

### API接口设计
- `POST /api/login` - 用户登录
- `POST /api/attendance/verify` - 考勤验证
- `POST /api/group/upload` - 合照上传
- `GET /api/students` - 学生列表
- `GET /api/stats/overview` - 统计数据

## 模块详细介绍

### 1. 人脸考勤模块

人脸考勤是系统的核心功能，实现从图像到身份识别的完整流程。

#### 算法流程
1. **图像预处理**：将摄像头帧从BGR转换为RGB格式
2. **人脸检测**：使用dlib HOG + CNN检测器定位人脸位置
3. **关键点提取**：对检测到的人脸提取68个面部关键点
4. **特征提取**：使用ResNet模型计算128维人脸特征向量
5. **相似度匹配**：计算查询向量与数据库中所有学生向量的欧氏距离
6. **身份确认**：根据距离阈值判断是否匹配成功

#### 输入输出
- **输入**：RGB图像 (numpy array, shape: H×W×3)
- **输出**：匹配的学生ID和相似度分数 (student_id: int, distance: float)

#### 关键算法公式

**欧氏距离计算**：
$$d(p,q) = \sqrt{\sum_{i=1}^{128} (p_i - q_i)^2}$$

**相似度匹配**：
- 距离越小，相似度越高
- 阈值通常设为0.4：如果d < 0.4则匹配成功

#### 流程图

```
图像输入
    ↓
人脸检测 (dlib detector)
    ↓
关键点提取 (shape_predictor)
    ↓
特征编码 (face_recognition_model)
    ↓
数据库查询 (1:N匹配)
    ↓
阈值判断
    ↓
输出结果 (学生ID + 置信度)
```

#### 代码流程
1. **图像预处理**：`face_engine.py:decode_image()` 将base64转换为BGR数组
2. **人脸检测**：`face_engine.py:detect_faces()` 调用dlib检测器
3. **特征提取**：`face_engine.py:encode_face()` 计算128维向量
4. **相似度匹配**：`face_engine.py:match_one()` 计算欧氏距离
5. **结果输出**：返回匹配学生ID和置信度分数

**数据流**：图像 → 人脸框列表 → 特征向量 → 距离计算 → 身份结果

### 2. 活体检测模块

活体检测采用双重验证机制，防止照片或视频重放攻击。

#### 2.1 动作活体检测（眨眼、张嘴、转头）

基于dlib 68关键点计算面部动作特征，实现无模型的活体验证。

##### 算法原理

**Eye Aspect Ratio (EAR)**：
眼睛纵横比，用于检测眨眼动作。

$$EAR = \frac{||p_2 - p_6|| + ||p_3 - p_5||}{2||p_1 - p_4||}$$

其中p1-p6为眼睛关键点。

**Mouth Aspect Ratio (MAR)**：
嘴巴纵横比，用于检测张嘴动作。

$$MAR = \frac{||p_{63} - p_{67}||}{||p_{61} - p_{65}||}$$

**Head Pose (Yaw)**：
使用solvePnP计算头部旋转角度。

##### 检测流程
1. 对每帧提取68关键点
2. 计算EAR、MAR、Yaw
3. 在帧序列上检测动作模式：
   - 眨眼：EAR从>0.27降到<0.21再回到>0.27
   - 张嘴：MAR峰值>0.55且与基线差>0.15
   - 转头：Yaw绝对值超过18°

##### 输入输出
- **输入**：RGB图像序列 (List[np.ndarray])
- **输出**：动作验证结果 (bool)

##### 流程图

```
帧序列输入
    ↓
逐帧关键点提取 (68点)
    ↓
计算特征: EAR/MAR/Yaw
    ↓
动作状态机检测
    ↓
阈值判断
    ↓
输出验证结果
```

#### 代码流程
1. **帧分析**：`liveness.py:measure_frame()` 计算EAR/MAR/Yaw特征
2. **眨眼检测**：`liveness.py:verify_action("blink")` 检查EAR变化模式
3. **张嘴检测**：`liveness.py:verify_action("open_mouth")` 检查MAR峰值
4. **转头检测**：`liveness.py:verify_action("turn_left/right")` 检查Yaw角度

**数据流**：帧序列 → 特征序列(EAR/MAR/Yaw) → 动作状态机 → 验证结果

#### 2.2 ClarityEye轻量活体检测模型

基于PyTorch的EfficientNet模型，进行真人/伪造二分类。

##### 模型架构
- **骨干网络**：EfficientNet-B0
- **输入尺寸**：224×224×3 (RGB人脸图像)
- **输出**：单值概率 (0-1)，>0.5为真人
- **训练数据**：真实人脸 + 打印照片/屏幕重放等攻击样本

##### 算法流程
1. **人脸对齐**：检测人脸框，裁剪并resize到224×224
2. **图像预处理**：归一化 (mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
3. **特征提取**：EfficientNet前向传播
4. **分类输出**：Sigmoid激活得到活体概率
5. **后处理**：多帧投票，取平均分数

##### 输入输出
- **输入**：人脸图像序列 (List[np.ndarray], RGB, H×W×3)
- **输出**：活体检测结果 (dict: passed, score, reason)

##### 关键公式

**图像归一化**：
$$x_{norm} = \frac{x - \mu}{\sigma}$$

其中 μ = [0.485, 0.456, 0.406], σ = [0.229, 0.224, 0.225]

**活体分数**：
$$score = \sigma(W \cdot f + b)$$

其中f为EfficientNet特征，σ为Sigmoid函数

**多帧投票**：
$$final\_score = \frac{1}{N} \sum_{i=1}^{N} score_i$$

##### 流程图

```
人脸图像序列
    ↓
人脸检测 + 裁剪
    ↓
Resize到224×224 + 归一化
    ↓
EfficientNet前向传播
    ↓
Sigmoid输出概率
    ↓
多帧平均投票
    ↓
阈值判断 (默认0.5)
    ↓
输出活体结果
```

#### 代码流程
1. **帧选择**：`liveness_model.py:pick_face_frame_indices()` 选择有效人脸帧
2. **预处理**：`liveness_model.py:_prepare_face_tensor()` 裁剪+归一化
3. **推理**：`liveness_model.py:score_frame()` EfficientNet前向传播
4. **投票**：`liveness_model.py:verify_liveness_from_frames()` 多帧平均

**数据流**：帧序列 → 人脸裁剪 → 张量预处理 → 模型推理 → 分数平均 → 活体判断

### 3. 表情检测模块

使用DeepFace库进行7类情绪识别：angry, disgust, fear, happy, sad, surprise, neutral。

#### 算法原理
DeepFace使用预训练的深度学习模型对人脸图像进行情绪分类。

##### 支持的情绪类别
1. **Angry** (愤怒) - 眉毛下压，眼睛瞪大，嘴巴紧闭
2. **Disgust** (厌恶) - 鼻子皱起，上唇抬起
3. **Fear** (恐惧) - 眼睛睁大，眉毛抬起，嘴巴张开
4. **Happy** (开心) - 嘴角上扬，眼睛弯成月牙
5. **Sad** (悲伤) - 嘴角下垂，眉毛内侧下垂
6. **Surprise** (惊讶) - 眼睛睁大，眉毛抬起，嘴巴张开
7. **Neutral** (中性) - 面部放松，无明显表情

##### 算法流程
1. **人脸检测**：使用预训练检测器定位人脸
2. **人脸对齐**：基于关键点进行几何变换
3. **特征提取**：CNN网络提取面部特征
4. **分类器**：Softmax输出7类概率
5. **后处理**：选择置信度最高的情绪

##### 输入输出
- **输入**：人脸图像 (np.ndarray, BGR格式)
- **输出**：情绪标签和置信度 (str, dict)

##### 关键公式

**Softmax分类**：
$$P(class_i) = \frac{e^{z_i}}{\sum_{j=1}^{7} e^{z_j}}$$

其中z_i为第i类原始分数。

**置信度最高的类别**：
$$emotion = \arg\max_{i} P(class_i)$$

##### 流程图

```
人脸图像输入
    ↓
人脸检测 (MTCNN/RetinaFace)
    ↓
人脸对齐 (关键点几何变换)
    ↓
CNN特征提取 (VGG/FaceNet)
    ↓
情绪分类器 (7类Softmax)
    ↓
输出情绪结果 + 置信度
```

##### 代码实现示例

### 4. 合照检测与人脸超分模块

本模块包含两个功能：多人合照批量识别和低质量人脸超分辨率重建。

#### 4.1 合照检测

处理多人合照场景，批量识别参与学生，避免重复考勤。

##### 算法原理
合照检测需要处理多个目标，同时考虑人脸质量和聚类。

**核心挑战**：
- **多人检测**：合照中可能有10-50人，需要高召回率检测
- **质量过滤**：远距离/侧脸人脸可能无法准确识别
- **去重处理**：同一人可能被检测多次
- **效率优化**：批量处理大量人脸

##### 算法流程
1. **图像预处理**：调整图像尺寸，提高检测精度
2. **多人脸检测**：使用更高upsample参数检测小脸
3. **质量评估**：过滤低质量/侧脸人脸
4. **特征提取**：对有效人脸计算特征向量
5. **身份匹配**：1:N匹配所有学生
6. **结果聚合**：统计识别结果，去重处理

##### 输入输出
- **输入**：合照图像 (np.ndarray, BGR)
- **输出**：识别结果列表 (List[dict]: student_id, confidence, bbox)

##### 关键公式

**人脸质量评估**：
$$quality = \frac{face\_area}{image\_area} \times confidence$$

**相似度阈值过滤**：
只保留相似度 > 阈值的结果，避免误识别。

##### 流程图

```
合照图像输入
    ↓
图像缩放 (提高检测精度)
    ↓
多人脸检测 (upsample=2)
    ↓
质量过滤 (面积/角度)
    ↓
批量特征提取
    ↓
数据库1:N匹配
    ↓
相似度阈值过滤
    ↓
去重聚合
    ↓
输出参与名单
```

##### 代码流程
1. **人脸检测**：`face_engine.py:detect_faces()` 使用upsample=2
2. **质量过滤**：计算face_area/image_area比例
3. **批量特征提取**：`face_engine.py:encode_face()` 对有效人脸
4. **身份匹配**：`face_engine.py:match_one()` 1:N匹配
5. **去重聚合**：统计唯一学生ID

**数据流**：合照 → 人脸框列表 → 质量过滤 → 特征向量列表 → 匹配结果 → 学生名单

#### 4.2 人脸超分

使用GFPGAN模型对低质量人脸进行超分辨率重建，提高识别精度。

##### 算法原理
GFPGAN (Generative Facial Prior GAN) 是一种专门针对人脸的超分模型。

**模型特点**：
- **生成对抗网络**：Generator + Discriminator
- **面部先验**：利用预训练的面部关键点和解析图
- **盲超分**：无需知道降质类型
- **2x放大**：将低分辨率人脸放大到高清

##### 算法流程
1. **人脸检测**：定位需要超分的人脸区域
2. **图像裁剪**：提取人脸ROI，添加margin
3. **超分重建**：GFPGAN生成高清人脸
4. **后处理**：融合回原图或单独输出

##### 输入输出
- **输入**：低分辨率人脸图像 (np.ndarray, BGR)
- **输出**：高清重建人脸 (np.ndarray, BGR)

##### 关键技术

**生成器架构**：
- U-Net结构
- 面部组件分支 (眼睛、鼻子、嘴巴)
- 纹理增强分支

**对抗训练**：
$$L = L_{rec} + \lambda L_{adv} + \mu L_{gp}$$

##### 流程图

```
低分辨率人脸输入
    ↓
人脸检测 + 裁剪 (添加margin)
    ↓
GFPGAN生成器前向传播
    ↓
多尺度特征融合
    ↓
纹理细节增强
    ↓
输出高清人脸
```

##### 代码流程
1. **人脸裁剪**：`face_engine.py:crop_face()` 提取人脸区域
2. **超分增强**：`face_enhancer.py:FaceEnhancer.enhance()` GFPGAN重建
3. **质量评估**：计算方差和SSIM指标

**数据流**：低清人脸 → 裁剪ROI → GFPGAN超分 → 高清重建结果

### 5. 学生管理与权限划分

本系统通过接口和装饰器区分教师与学生角色，并支持学生录入、批量导入、账号生成与管理。

#### 5.1 学生账号与导入流程

学生管理的核心逻辑在 `app/routes/api_students.py`：
- `add_student()`：教师上传单个学生照片，创建学生记录
- `batch_import()`：教师批量导入多个学生照片，使用文件名解析 `学号_姓名_班级.jpg`
- `create_account()`：教师为已存在学生生成/恢复登录账号

关键辅助函数：
- `parse_student_filename()`：解析批量导入文件名
- `add_or_update_student_from_image()`：从照片提取人脸特征并保存学生信息
- `_create_student_account()`：默认用户名为学号，默认密码为学号后 6 位或 `123456`

#### 5.2 权限划分与角色控制

权限逻辑在 `app/auth.py` 中定义：
- `login_required`：用户登录后才能访问接口
- `teacher_required`：仅教师角色可访问管理接口

部分 API 权限示例：
- `app/routes/api_students.py`：学生 CRUD、批量导入、账号创建均使用 `@teacher_required`
- `app/routes/api_group.py`：合照上传和活动管理也要求教师权限
- `app/routes/api_attendance.py`：考勤验证与记录查询要求登录即可访问
- `app/routes/auth_routes.py`：`/api/login`、`/api/logout`、`/api/me` 提供会话管理

#### 5.3 学生自查与教师管理的区别

- 教师登录后可查看和管理所有学生记录；
- 学生登录后通过 `api_students.py:list_students()` 仅能查看自己的学生信息；
- 学生账号默认映射到 `students.student_id`，用于限定自己的考勤和情绪数据。

#### 5.4 代码路径总结

- `app/routes/api_students.py`：学生信息增删改查、批量导入、账号创建
- `app/auth.py`：登录、登出、权限装饰器、当前用户会话
- `app/routes/auth_routes.py`：登录接口与教师创建用户接口
- `app/routes/api_attendance.py`：考勤验证、考勤记录查询、Excel 导出
- `app/routes/api_group.py`：合照识别、活动分析、超分样本保存

**整体数据流**：教师上传学生照片/合照 → 后端人脸特征提取与保存 → 教师发起考勤或活动识别 → 系统根据权限写入记录并返回结果

### 核心扩展

本节补全了项目中“学生管理”与“权限划分”的真实路径，便于你在 notebook 中直接对照代码。

### 核心模块总结

| 模块 | 算法 | 输入 | 输出 | 关键参数 |
|------|------|------|------|----------|
| **人脸考勤** | dlib + 特征匹配 | RGB图像 | 学生ID + 距离 | 阈值=0.6 |
| **动作活体** | EAR/MAR + PnP | 帧序列 | 动作验证结果 | EAR阈值=0.21/0.27 |
| **模型活体** | EfficientNet | 人脸序列 | 活体分数 | 阈值=0.5 |
| **情绪检测** | DeepFace | 人脸图像 | 7类情绪 | 置信度 |
| **合照检测** | 批量识别 | 合照图像 | 学生名单 | upsample=2 |
| **人脸超分** | GFPGAN | 低清人脸 | 高清人脸 | 2x放大 |